### Cartesian radiating sphere test

Time-averaged flux factor $f = F_r / E$ and Eddington factor $p = P_{rr} / E$ along the outward $+x$ ray from the sphere center, compared with the analytic radiating-sphere solution.

In [ ]:
import os
import sys
from pathlib import Path
import glob

import matplotlib.pyplot as plt
import h5py
import numpy as np

try:
    here = Path(__file__).resolve().parent
except NameError:
    here = Path.cwd()

sys.path.append(str((here / '..' / 'plot_functions').resolve()))

import plot_functions as pf

plots_dir = here / 'plots'
plots_dir.mkdir(exist_ok=True)

Simulation and test parameters

In [ ]:
# Directory with mesh_plt*.h5 dumps
directory_path = (
    '/pscratch/sd/b/bigqke/bigqke_scratch/'
    'emu_methods_paper_simulations/bigqke_per/radiating_sphere_test'
)

# Cartesian grid size (checked against each dump)
Nx, Ny, Nz = 96, 96, 64

# Sphere-center cell indices (checked against the grid)
r_xi, r_yi, r_zi = 48, 48, 32

# Physics / comparison parameters (same as the 1D spherical test)
R = 1.17e6          # sphere radius, cm
kappa_R = 10.0009
n_mu = 2000         # number of integration bins for the analytic solution
tolerance = 0.07
r_min_over_R = 0.6  # inner radius cutoff for the error test

Read mesh dumps and extract the outward $+x$ ray

In [ ]:
filenames = sorted(
    glob.glob(os.path.join(directory_path, 'mesh_plt*.h5')),
    key=lambda x: int(os.path.basename(x).split('plt')[1].split('.')[0]),
)

if len(filenames) == 0:
    raise FileNotFoundError(f'No mesh_plt*.h5 files found in {directory_path}')

E_dumps, Fr_dumps, Prr_dumps = [], [], []
dx = dy = dz = None

for filename in filenames:
    with h5py.File(filename, 'r') as hf:
        file_Nx = int(np.array(hf['Nx']))
        file_Ny = int(np.array(hf['Ny']))
        file_Nz = int(np.array(hf['Nz']))
        if (file_Nx, file_Ny, file_Nz) != (Nx, Ny, Nz):
            raise ValueError(
                f'{os.path.basename(filename)} has grid '
                f'({file_Nx}, {file_Ny}, {file_Nz}), expected ({Nx}, {Ny}, {Nz})'
            )

        E = np.array(hf['N00_Re(1|ccm)'], dtype=float)
        Fx = np.array(hf['Fx00_Re(1|ccm)'], dtype=float)
        Pxx = np.array(hf['Pxx00_Re(1|ccm)'], dtype=float)

        if E.shape != (Nx, Ny, Nz):
            raise ValueError(
                f'{os.path.basename(filename)} field shape {E.shape} '
                f'does not match ({Nx}, {Ny}, {Nz})'
            )

        dx = float(np.array(hf['dx(cm)']))
        dy = float(np.array(hf['dy(cm)']))
        dz = float(np.array(hf['dz(cm)']))

        # Outward radial direction = +x through the sphere center
        E_dumps.append(E[r_xi:, r_yi, r_zi])
        Fr_dumps.append(Fx[r_xi:, r_yi, r_zi])
        Prr_dumps.append(Pxx[r_xi:, r_yi, r_zi])

if not (0 <= r_xi < Nx and 0 <= r_yi < Ny and 0 <= r_zi < Nz):
    raise ValueError(
        f'Sphere-center cell ({r_xi}, {r_yi}, {r_zi}) is outside '
        f'grid ({Nx}, {Ny}, {Nz})'
    )

E_all = np.array(E_dumps, dtype=float)      # (ndump, nray)
Fr_all = np.array(Fr_dumps, dtype=float)
Prr_all = np.array(Prr_dumps, dtype=float)

# Cell centers along x; radial distance measured from the sphere-center cell
x_centers = (np.arange(Nx) + 0.5) * dx
r_centers = x_centers[r_xi:] - x_centers[r_xi]

print(f'Read {len(filenames)} mesh dumps from {directory_path}')
print(f'Grid (Nx, Ny, Nz) = ({Nx}, {Ny}, {Nz})')
print(f'dx, dy, dz = {dx:.6g}, {dy:.6g}, {dz:.6g} cm')
print(f'Sphere-center cell (r_xi, r_yi, r_zi) = ({r_xi}, {r_yi}, {r_zi})')
print(f'Ray length = {len(r_centers)} cells along +x')

Time-averaged simulation flux factor and Eddington factor

In [ ]:
# Empty cells are NaN and drop out of that cell's average rather than counting as zero.
E_all = E_all.copy()
E_all[E_all <= 0.0] = np.nan
f_simulation = np.nanmean(Fr_all / E_all, axis=0)
p_simulation = np.nanmean(Prr_all / E_all, axis=0)

Analytic radiating-sphere solution

In [ ]:
f_analytic = np.zeros(len(r_centers))
p_analytic = np.zeros(len(r_centers))

for i, ri in enumerate(r_centers):
    if ri < R:
        mu_arr = np.linspace(-1.0, 1.0, n_mu)
        g = np.sqrt(np.maximum(1.0 - (ri / R)**2 * (1.0 - mu_arr**2), 0.0))
        s = ri * mu_arr / R + g
    else:
        mu_c = np.sqrt(1.0 - (R / ri)**2)
        mu_arr = np.linspace(mu_c, 1.0, n_mu)
        g = np.sqrt(np.maximum(1.0 - (ri / R)**2 * (1.0 - mu_arr**2), 0.0))
        s = 2 * g

    F_mu = 1.0 - np.exp(-kappa_R * s)

    E_int = np.trapezoid(F_mu, mu_arr)
    f_analytic[i] = np.trapezoid(mu_arr * F_mu, mu_arr) / E_int
    p_analytic[i] = np.trapezoid(mu_arr**2 * F_mu, mu_arr) / E_int

Error estimate

In [ ]:
valid = np.isfinite(f_simulation) & (r_centers >= r_min_over_R * R)

f_error = np.abs(f_simulation[valid] - f_analytic[valid])
p_error = np.abs(p_simulation[valid] - p_analytic[valid])

f_max_error = np.max(f_error)
p_max_error = np.max(p_error)

r_valid = r_centers[valid]

print(
    f'Time averaged {E_all.shape[0]} mesh data dumps '
    f'for the region r/R >= {r_min_over_R}'
)
print(
    f'Max error in f: {f_max_error}  '
    f'at r/R = {r_valid[np.argmax(f_error)] / R:.4f}'
)
print(
    f'Max error in p: {p_max_error}  '
    f'at r/R = {r_valid[np.argmax(p_error)] / R:.4f}'
)

if f_max_error < tolerance and p_max_error < tolerance:
    print(f'SUCCESS (errors below tolerance {tolerance})')
else:
    print(f'FAIL (errors above tolerance {tolerance})')

Plot simulation vs analytic $f$ and $p$

In [ ]:
r_over_R = r_centers / R

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(r_over_R, f_simulation, 'o', markersize=4, label=r'$f_{\mathrm{sim}}$')
ax.plot(r_over_R, f_analytic, '-', linewidth=2, label=r'$f_{\mathrm{analytic}}$')
ax.plot(r_over_R, p_simulation, 's', markersize=4, label=r'$p_{\mathrm{sim}}$')
ax.plot(r_over_R, p_analytic, '--', linewidth=2, label=r'$p_{\mathrm{analytic}}$')
ax.axvline(1.0, color='k', linestyle=':', linewidth=1.5, label=r'$r = R$')
ax.axvline(r_min_over_R, color='gray', linestyle=':', linewidth=1.5,
           label=rf'$r/R = {r_min_over_R}$')
ax.set_xlabel(r'$r / R$')
ax.set_ylabel(r'$f,\, p$')
ax.set_xlim(0.0, None)
leg = ax.legend(framealpha=0.0, ncol=2, fontsize=16)
pf.apply_custom_settings(ax, leg, False)
outfile = plots_dir / f"{directory_path.rstrip('/').split('/')[-1]}_f_p_vs_r.pdf"
fig.savefig(outfile, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f'Saved {outfile}')

Absolute error versus $r/R$

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(r_over_R[valid], f_error, 'o-', markersize=4, label=r'$|f_{\mathrm{sim}} - f_{\mathrm{analytic}}|$')
ax.plot(r_over_R[valid], p_error, 's-', markersize=4, label=r'$|p_{\mathrm{sim}} - p_{\mathrm{analytic}}|$')
ax.axhline(tolerance, color='k', linestyle='--', linewidth=1.5,
           label=rf'tolerance $= {tolerance}$')
ax.axvline(1.0, color='k', linestyle=':', linewidth=1.5, label=r'$r = R$')
ax.set_xlabel(r'$r / R$')
ax.set_ylabel(r'absolute error')
leg = ax.legend(framealpha=0.0, ncol=1, fontsize=16)
pf.apply_custom_settings(ax, leg, False)
outfile = plots_dir / f"{directory_path.rstrip('/').split('/')[-1]}_f_p_error_vs_r.pdf"
fig.savefig(outfile, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f'Saved {outfile}')